# 02b — Update Dataset to April 2026

Дополняет существующий датасет свежими данными из yfinance (OHLCV, macro) и RSS (news sentiment).

**Новый split:**
- Train: → 2025-06-30
- Val: 2025-07-01 → 2025-10-31
- Test: 2025-11-01 →

In [1]:
!pip install -q pytorch-forecasting==1.1.1 lightning numpy==1.26.4 pandas==2.1.4 yfinance feedparser transformers
from google.colab import drive
drive.mount('/content/drive')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 22.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.

In [1]:
import os, json, pickle, time, warnings, gc
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import torch
import yfinance as yf
import feedparser
from datetime import datetime, timedelta
from tqdm.notebook import tqdm
from transformers import AutoTokenizer, AutoModel

DRIVE_DATA = '/content/drive/MyDrive/predictamarket/data'
MODEL_DIR = '/content/drive/MyDrive/predictamarket/models'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

Device: cuda


## Load existing data

In [ ]:
# Get current S&P 500 tickers
import urllib.request
from io import StringIO

print('Fetching current S&P 500 list...')
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
html = urllib.request.urlopen(req).read().decode('utf-8')
sp500_table = pd.read_html(StringIO(html))[0]

sp500_tickers = set(sp500_table['Symbol'].str.replace('.', '-', regex=False).str.lower().tolist())
sector_map_wiki = dict(zip(
    sp500_table['Symbol'].str.replace('.', '-', regex=False).str.lower(),
    sp500_table['GICS Sector']
))

with open(os.path.join(DRIVE_DATA, 'config.json')) as f:
    config = json.load(f)
with open(os.path.join(DRIVE_DATA, 'pca_model.pkl'), 'rb') as f:
    pca = pickle.load(f)

dataset_tickers = set(config['tickers'])
sectors_old = config.get('sectors', {})

overlap_tickers = sorted(sp500_tickers & dataset_tickers)
remaining_dataset = sorted(dataset_tickers - sp500_tickers)

print(f'S&P 500: {len(sp500_tickers)}')
print(f'Dataset: {len(dataset_tickers)}')
print(f'Overlap (S&P 500 ∩ dataset): {len(overlap_tickers)}')

need_extra = 400 - len(overlap_tickers)
print(f'Need {need_extra} more from existing dataset')

# Load existing data to count rows per ticker
import pyarrow.parquet as pq

dfs = []
for fname in ['train.parquet', 'val.parquet', 'test.parquet']:
    fpath = os.path.join(DRIVE_DATA, fname)
    if os.path.exists(fpath):
        table = pq.read_table(fpath, columns=['ticker', 'Date'])
        dfs.append(table.to_pandas())
        del table

ticker_counts_df = pd.concat(dfs, ignore_index=True)
del dfs; gc.collect()

ticker_counts = ticker_counts_df.groupby('ticker').size().sort_values(ascending=False)
del ticker_counts_df; gc.collect()

extra_tickers = [t for t in ticker_counts.index if t in remaining_dataset][:need_extra]
print(f'Extra tickers (best data quality from dataset): {len(extra_tickers)}')
print(f'  Min rows: {ticker_counts[extra_tickers[-1]] if extra_tickers else 0}')
print(f'  Max rows: {ticker_counts[extra_tickers[0]] if extra_tickers else 0}')

top_400 = sorted(overlap_tickers + extra_tickers)
print(f'\nFinal selection: {len(top_400)} tickers')
print(f'  From S&P 500: {len(overlap_tickers)}')
print(f'  Extra from dataset: {len(extra_tickers)}')

sector_map = {}
for t in top_400:
    if t in sector_map_wiki:
        sector_map[t] = sector_map_wiki[t]
    elif t in sectors_old:
        sector_map[t] = sectors_old[t]
    else:
        sector_map[t] = 'N/A'

dfs = []
for fname in ['train.parquet', 'val.parquet', 'test.parquet']:
    fpath = os.path.join(DRIVE_DATA, fname)
    if os.path.exists(fpath):
        table = pq.read_table(fpath, filters=[('ticker', 'in', top_400)])
        df = table.to_pandas()
        float_cols = df.select_dtypes(include=['float64']).columns
        df[float_cols] = df[float_cols].astype(np.float32)
        dfs.append(df)
        del table, df

old_df = pd.concat(dfs, ignore_index=True)
del dfs; gc.collect()

old_df['Date'] = pd.to_datetime(old_df['Date'])
print(f'\nLoaded existing data: {old_df.shape}')
print(f'Date range: {old_df["Date"].min().date()} to {old_df["Date"].max().date()}')
print(f'Tickers: {old_df["ticker"].nunique()}')
print(f'Memory: {old_df.memory_usage(deep=True).sum() / 1e9:.1f} GB')

Fetching current S&P 500 list...
S&P 500: 503
Dataset: 400
Overlap (S&P 500 ∩ dataset): 347
Need 53 more from existing dataset
Extra tickers (best data quality from dataset): 53
  Min rows: 6553
  Max rows: 6553

Final selection: 400 tickers
  From S&P 500: 347
  Extra from dataset: 53

Loaded existing data: (2393137, 115)
Date range: 2000-03-14 to 2026-04-02
Tickers: 400
Memory: 1.6 GB


## Download fresh OHLCV (yfinance)

In [ ]:
def compute_rsi(series, period=14):
    delta = series.diff()
    gain = delta.where(delta > 0, 0.0)
    loss = -delta.where(delta < 0, 0.0)
    avg_gain = gain.rolling(window=period).mean()
    avg_loss = loss.rolling(window=period).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

def process_ticker_df(df, ticker):
    """Add technical features to a ticker DataFrame."""
    df['log_return'] = np.log(df['Close'] / df['Close'].shift(1))
    df['volatility_5d'] = df['log_return'].rolling(5).std()
    df['volatility_20d'] = df['log_return'].rolling(20).std()
    df['ma_5'] = df['Close'].rolling(5).mean()
    df['ma_20'] = df['Close'].rolling(20).mean()
    df['ma_50'] = df['Close'].rolling(50).mean()
    df['rsi_14'] = compute_rsi(df['Close'], 14)
    df['volume_ma_20'] = df['Volume'].rolling(20).mean()
    df['price_to_ma20'] = df['Close'] / df['ma_20']
    df['price_to_ma50'] = df['Close'] / df['ma_50']
    df['momentum_5d'] = df['Close'].pct_change(5)
    df['momentum_20d'] = df['Close'].pct_change(20)
    df['pct_from_52wk_high'] = df['Close'] / df['Close'].rolling(252).max() - 1
    df['pct_from_52wk_low'] = df['Close'] / df['Close'].rolling(252).min() - 1
    df['ticker'] = ticker
    df['sector'] = sector_map.get(ticker, 'N/A')
    df['day_of_week'] = df['Date'].dt.dayofweek.astype(str)
    df['month'] = df['Date'].dt.month.astype(str)
    df = df.dropna(subset=['ma_50']).reset_index(drop=True)
    float_cols = df.select_dtypes(include=['float64']).columns
    df[float_cols] = df[float_cols].astype(np.float32)
    return df

# All 400 tickers already have data — download only the new period
old_max_date = old_df['Date'].max()
overlap_start = (old_max_date - pd.Timedelta(days=90)).strftime('%Y-%m-%d')
end_date = '2026-04-04'

print(f'Downloading new data from {overlap_start} to {end_date}')

new_ticker_dfs = []
errors = []

for ticker in tqdm(top_400, desc='Downloading OHLCV updates'):
    try:
        df = yf.download(ticker, start=overlap_start, end=end_date, progress=False)
        if len(df) == 0:
            errors.append(ticker)
            continue
        df = df.reset_index()
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]
        df['Date'] = pd.to_datetime(df['Date']).dt.tz_localize(None).dt.normalize()
        df = df.sort_values('Date').drop(columns=['Adj Close'], errors='ignore').reset_index(drop=True)
        df = process_ticker_df(df, ticker)
        new_ticker_dfs.append(df)
    except:
        errors.append(ticker)

print(f'\nDownloaded: {len(new_ticker_dfs)} tickers')
print(f'Errors: {len(errors)}')
if errors[:10]:
    print(f'  Examples: {errors[:10]}')

new_ohlcv = pd.concat(new_ticker_dfs, ignore_index=True)
del new_ticker_dfs; gc.collect()
print(f'New OHLCV: {new_ohlcv.shape}')
print(f'Date range: {new_ohlcv["Date"].min().date()} to {new_ohlcv["Date"].max().date()}')


Downloaded: 400 tickers
Errors: 0
New OHLCV: (5600, 24)
Date range: 2026-03-16 to 2026-04-02


In [ ]:
last_dates = old_df.groupby('ticker')['Date'].max()
print(f'Median last date: {last_dates.median().date()}')
print(f'Min last date: {last_dates.min().date()}')
print(f'Max last date: {last_dates.max().date()}')
print(f'\nТикеров до 2025+: {(last_dates >= "2025-01-01").sum()}')
print(f'Тикеров до 2024 только: {(last_dates < "2025-01-01").sum()}')


Median last date: 2026-04-02
Min last date: 2026-04-02
Max last date: 2026-04-02

Тикеров до 2025+: 400
Тикеров до 2024 только: 0


## Download fresh macro

In [ ]:
start_date = overlap_start
MACRO_MAP = {
    '^VIX': 'vix', '^TNX': 'treasury_10y', '^GSPC': 'sp500',
    'DX-Y.NYB': 'dxy', 'GC=F': 'gold', 'CL=F': 'oil',
}

print('Fetching macro data...')
macro_dfs = []
for symbol, name in MACRO_MAP.items():
    try:
        data = yf.download(symbol, start=start_date, end=end_date, progress=False)
        if len(data) > 0:
            series = data['Close'].squeeze().rename(name)
            macro_dfs.append(series)
            print(f'  {symbol} ({name}): {len(data)} rows')
    except Exception as e:
        print(f'  {symbol}: ERROR - {e}')

macro = pd.concat(macro_dfs, axis=1).ffill()
macro.index = macro.index.tz_localize(None) if macro.index.tz else macro.index

if 'vix' in macro.columns:
    macro['vix_ma5'] = macro['vix'].rolling(5).mean()
if 'sp500' in macro.columns:
    macro['sp500_return'] = np.log(macro['sp500'] / macro['sp500'].shift(1))

try:
    vix3m = yf.download('^VIX3M', start=start_date, end=end_date, progress=False)['Close'].squeeze()
    vix3m.index = vix3m.index.tz_localize(None) if vix3m.index.tz else vix3m.index
    macro['vix_contango'] = vix3m / macro['vix'] - 1
except:
    macro['vix_contango'] = 0.0

macro = macro.dropna().reset_index()
if 'index' in macro.columns:
    macro = macro.rename(columns={'index': 'Date'})
if 'Date' not in macro.columns:
    macro = macro.rename(columns={macro.columns[0]: 'Date'})
macro['Date'] = pd.to_datetime(macro['Date'])
print(f'Macro: {macro.shape}')

# Merge macro into new OHLCV
new_ohlcv = new_ohlcv.merge(macro, on='Date', how='left')
for col in macro.columns:
    if col != 'Date' and col in new_ohlcv.columns:
        new_ohlcv[col] = new_ohlcv[col].ffill().bfill()
print(f'After macro merge: {new_ohlcv.shape}')

Fetching macro data...
  ^VIX (vix): 63 rows
  ^TNX (treasury_10y): 63 rows
  ^GSPC (sp500): 63 rows
  DX-Y.NYB (dxy): 64 rows
  GC=F (gold): 63 rows
  CL=F (oil): 63 rows
Macro: (59, 10)
After macro merge: (5600, 33)


## News sentiment (FinBERT + PCA)

In [6]:
# Load FinBERT
tokenizer = AutoTokenizer.from_pretrained('ProsusAI/finbert')
finbert = AutoModel.from_pretrained('ProsusAI/finbert').to(DEVICE)
finbert.eval()
print('FinBERT loaded.')

n_components = config.get('n_sentiment_components', 32)
sent_cols = [f'sent_{i}' for i in range(n_components)] + ['news_count']

all_sentiment = []

for ticker in tqdm(top_400, desc='News Sentiment'):
    try:
        # Fetch RSS
        url = f'https://feeds.finance.yahoo.com/rss/2.0/headline?s={ticker}&region=US&lang=en-US'
        feed = feedparser.parse(url)

        texts, dates = [], []
        for entry in feed.entries[:30]:
            text = (entry.get('title', '') + '. ' + entry.get('summary', ''))[:512]
            try:
                date = pd.to_datetime(entry.get('published', '')).tz_localize(None).normalize()
            except:
                date = pd.Timestamp.now().normalize()
            if isinstance(text, str) and text.strip():
                texts.append(text)
                dates.append(date)

        if not texts:
            continue

        # FinBERT embeddings
        all_emb = []
        for i in range(0, len(texts), 16):
            batch = texts[i:i+16]
            inputs = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors='pt').to(DEVICE)
            with torch.no_grad():
                outputs = finbert(**inputs)
                cls = outputs.last_hidden_state[:, 0, :].cpu().numpy()
                all_emb.append(cls)
            del inputs, outputs

        embeddings = np.vstack(all_emb)
        reduced = pca.transform(embeddings)

        # Aggregate daily
        df = pd.DataFrame(reduced, columns=[f'sent_{i}' for i in range(n_components)])
        df['Date'] = dates
        df['ticker'] = ticker
        daily = df.groupby(['Date', 'ticker']).agg(
            **{f'sent_{i}': (f'sent_{i}', 'mean') for i in range(n_components)},
            news_count=('Date', 'count')
        ).reset_index()
        all_sentiment.append(daily)
    except:
        pass

if all_sentiment:
    sentiment_df = pd.concat(all_sentiment, ignore_index=True)
    print(f'Sentiment: {len(sentiment_df)} daily records, {sentiment_df["ticker"].nunique()} tickers')

    # Merge into new OHLCV
    new_ohlcv = new_ohlcv.merge(sentiment_df, on=['Date', 'ticker'], how='left')
else:
    print('No sentiment data fetched')

# Fill missing sentiment with 0
for col in sent_cols:
    if col not in new_ohlcv.columns:
        new_ohlcv[col] = 0.0
    else:
        new_ohlcv[col] = new_ohlcv[col].fillna(0.0)

# Free GPU memory
del finbert, tokenizer
torch.cuda.empty_cache(); gc.collect()
print(f'After sentiment: {new_ohlcv.shape}')

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
classifier.bias              | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 
classifier.weight            | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FinBERT loaded.


News Sentiment:   0%|          | 0/400 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Sentiment: 4233 daily records, 400 tickers
After sentiment: (5600, 66)


## Merge new + existing data

In [ ]:
for col in old_df.columns:
    if col not in new_ohlcv.columns:
        new_ohlcv[col] = 0.0

common_cols = [c for c in old_df.columns if c in new_ohlcv.columns]
new_ohlcv = new_ohlcv[common_cols]

full_df = pd.concat([old_df, new_ohlcv], ignore_index=True)
del old_df, new_ohlcv; gc.collect()

full_df['Date'] = pd.to_datetime(full_df['Date'])
full_df = full_df.drop_duplicates(subset=['ticker', 'Date'], keep='last')
full_df = full_df.sort_values(['ticker', 'Date']).reset_index(drop=True)

full_df.replace([np.inf, -np.inf], np.nan, inplace=True)
full_df.dropna(subset=['Close'], inplace=True)
full_df.drop(full_df[full_df['Close'] <= 0].index, inplace=True)

all_dates = sorted(full_df['Date'].unique())
date_to_idx = {d: i for i, d in enumerate(all_dates)}
full_df['time_idx'] = full_df['Date'].map(date_to_idx)

for col in ['ticker', 'sector', 'day_of_week', 'month']:
    if col in full_df.columns:
        full_df[col] = full_df[col].astype(str)

numeric_cols = full_df.select_dtypes(include=[np.number]).columns
full_df[numeric_cols] = full_df[numeric_cols].fillna(0.0)

print(f'Merged dataset: {full_df.shape}')
print(f'Date range: {full_df["Date"].min().date()} to {full_df["Date"].max().date()}')
print(f'Tickers: {full_df["ticker"].nunique()}')
print(f'Trading days: {len(all_dates)}')
print(f'Memory: {full_df.memory_usage(deep=True).sum() / 1e9:.1f} GB')

Merged dataset: (2393137, 115)
Date range: 2000-03-14 to 2026-04-02
Tickers: 400
Trading days: 6553
Memory: 2.5 GB


## Update missing data sources (FRED, Earnings, Insider, Calendar)

In [ ]:
new_period_mask = full_df['Date'] >= '2025-01-01'
print(f'Rows to update with additional sources: {new_period_mask.sum()}')

Rows to update with additional sources: 125200


In [ ]:
# FRED Macro Data
!pip install -q fredapi

try:
    from fredapi import Fred
    fred = Fred(api_key='eb14b7616b8071a075769fdad3158a50')

    fred_series = {
        'CPIAUCSL': 'cpi',
        'UNRATE': 'unemployment',
        'DFF': 'fed_funds_rate',
        'T10Y2Y': 'yield_curve_spread',
        'M2SL': 'm2_money_supply',
        'DCOILWTICO': 'wti_crude',
        'VIXCLS': 'fred_vix',
    }

    for series_id, name in fred_series.items():
        if name in full_df.columns:
            try:
                s = fred.get_series(series_id, observation_start='2025-01-01')
                s = s.dropna()
                date_val = dict(zip(pd.to_datetime(s.index).normalize(), s.values))
                mask = full_df['Date'].isin(date_val.keys())
                full_df.loc[mask, name] = full_df.loc[mask, 'Date'].map(date_val)
                full_df[name] = full_df[name].ffill()
                print(f'  {name}: updated {mask.sum()} rows')
            except Exception as e:
                print(f'  {name}: {e}')
    print('FRED done')
except Exception as e:
    print(f'FRED skipped: {e}')

  cpi: updated 2000 rows
  unemployment: updated 2000 rows
  fed_funds_rate: updated 125200 rows
  yield_curve_spread: updated 124400 rows
  m2_money_supply: updated 2400 rows
  wti_crude: updated 123200 rows
  fred_vix: updated 124800 rows
FRED done


In [ ]:
# VIX Term Structure
try:
    vix1 = yf.download('^VIX', start='2025-01-01', progress=False)['Close'].squeeze()
    vix3 = yf.download('^VIX3M', start='2025-01-01', progress=False)['Close'].squeeze()
    vix_term = pd.DataFrame({'vix_spot': vix1, 'vix_3m': vix3}).dropna()
    vix_term['vix_contango'] = vix_term['vix_3m'] / vix_term['vix_spot'] - 1
    vix_term.index = vix_term.index.tz_localize(None) if vix_term.index.tz else vix_term.index

    date_val = dict(zip(vix_term.index.normalize(), vix_term['vix_contango'].values))
    mask = full_df['Date'].isin(date_val.keys())
    full_df.loc[mask, 'vix_contango'] = full_df.loc[mask, 'Date'].map(date_val)
    full_df['vix_contango'] = full_df['vix_contango'].ffill()
    print(f'VIX contango: updated {mask.sum()} rows')
except Exception as e:
    print(f'VIX term error: {e}')

VIX contango: updated 125200 rows


In [ ]:
# Earnings (yfinance)
earnings_updates = 0
for ticker in tqdm(top_400, desc='Earnings'):
    try:
        tk = yf.Ticker(ticker)
        earnings = tk.earnings_dates
        if earnings is None or len(earnings) == 0:
            continue
        e = earnings.reset_index()
        e.columns = [c.strip() for c in e.columns]
        if 'Earnings Date' in e.columns:
            e['Date'] = pd.to_datetime(e['Earnings Date']).dt.tz_localize(None).dt.normalize()
        elif 'Date' in e.columns:
            e['Date'] = pd.to_datetime(e['Date']).dt.tz_localize(None).dt.normalize()
        else:
            continue
        e = e[e['Date'] >= '2025-01-01']
        if len(e) == 0:
            continue

        surprise_col = [c for c in e.columns if 'surprise' in c.lower() or 'Surprise' in c]
        if surprise_col:
            for _, row in e.iterrows():
                mask = (full_df['ticker'] == ticker) & (full_df['Date'] == row['Date'])
                if mask.sum() > 0:
                    val = pd.to_numeric(row[surprise_col[0]], errors='coerce')
                    if pd.notna(val):
                        full_df.loc[mask, 'eps_surprise_pct'] = val
                        full_df.loc[mask, 'earnings_beat'] = float(val > 0)
                        full_df.loc[mask, 'earnings_miss'] = float(val < 0)
                        full_df.loc[mask, 'has_earnings'] = 1.0
                        earnings_updates += 1
    except:
        pass

print(f'Earnings: {earnings_updates} updates')

Earnings:   0%|          | 0/400 [00:00<?, ?it/s]

ERROR:yfinance:AON: No earnings dates found, symbol may be delisted
ERROR:yfinance:CULP: No earnings dates found, symbol may be delisted
ERROR:yfinance:FITB: No earnings dates found, symbol may be delisted
ERROR:yfinance:NOW: No earnings dates found, symbol may be delisted


Earnings: 1917 updates


In [ ]:
# Insider Trading
insider_updates = 0
for ticker in tqdm(top_400[:200], desc='Insider Trading'):
    try:
        tk = yf.Ticker(ticker)
        insider = tk.insider_transactions
        if insider is None or len(insider) == 0:
            continue
        ins = insider.copy()

        date_col = None
        for c in ['Start Date', 'Date', 'date']:
            if c in ins.columns:
                date_col = c
                break
        if date_col is None:
            continue

        ins['Date'] = pd.to_datetime(ins[date_col], errors='coerce').dt.tz_localize(None).dt.normalize()
        ins = ins[ins['Date'] >= '2025-01-01'].dropna(subset=['Date'])
        if len(ins) == 0:
            continue

        shares_col = None
        for c in ['Shares', 'shares', 'Value']:
            if c in ins.columns:
                shares_col = c
                break

        if shares_col:
            for date, group in ins.groupby('Date'):
                shares = pd.to_numeric(group[shares_col], errors='coerce').fillna(0)
                mask = (full_df['ticker'] == ticker) & (full_df['Date'] == date)
                if mask.sum() > 0:
                    full_df.loc[mask, 'insider_buys'] = float((shares > 0).sum())
                    full_df.loc[mask, 'insider_sells'] = float((shares < 0).sum())
                    full_df.loc[mask, 'insider_net'] = float((shares > 0).sum() - (shares < 0).sum())
                    insider_updates += 1
    except:
        pass

print(f'Insider trading: {insider_updates} updates')

Insider Trading:   0%|          | 0/200 [00:00<?, ?it/s]

Insider trading: 4461 updates


In [ ]:
# Calendar Features (FOMC, Options Expiration
FOMC_2025_2026 = pd.to_datetime([
    '2025-01-29', '2025-03-19', '2025-05-07', '2025-06-18', '2025-07-30',
    '2025-09-17', '2025-10-29', '2025-12-17',
    '2026-01-28', '2026-03-18', '2026-05-06', '2026-06-17', '2026-07-29',
    '2026-09-16', '2026-10-28', '2026-12-16',
])

from pandas.tseries.offsets import WeekOfMonth
options_exp = pd.date_range('2025-01-01', '2026-12-31', freq=WeekOfMonth(week=2, weekday=4))
quad_witch = [d for d in options_exp if d.month in [3, 6, 9, 12]]

new_mask = full_df['Date'] >= '2025-01-01'
new_dates = full_df.loc[new_mask, 'Date']

days_to_fomc = []
for d in new_dates:
    if pd.notna(d):
        days_to_fomc.append(min((abs((d - fd).days) for fd in FOMC_2025_2026), default=999))
    else:
        days_to_fomc.append(999)

full_df.loc[new_mask, 'days_to_fomc'] = np.minimum(days_to_fomc, 30).astype(float)
full_df.loc[new_mask, 'is_options_expiration'] = new_dates.isin(options_exp).astype(float).values
full_df.loc[new_mask, 'is_quad_witching'] = new_dates.isin(quad_witch).astype(float).values

print(f'Calendar: updated {new_mask.sum()} rows')
print(f'FOMC dates: {len(FOMC_2025_2026)}')
print(f'\n=== All additional sources updated ===')

Calendar: updated 125200 rows
FOMC dates: 16

=== All additional sources updated ===


## New Train/Val/Test Split

In [ ]:
train_cutoff = pd.Timestamp('2025-06-30')
val_cutoff = pd.Timestamp('2025-10-31')

train = full_df[full_df['Date'] <= train_cutoff].copy()
val = full_df[(full_df['Date'] > train_cutoff) & (full_df['Date'] <= val_cutoff)].copy()
test = full_df[full_df['Date'] > val_cutoff].copy()

HALF_LIFE_DAYS = 504
max_idx = train['time_idx'].max()
train['sample_weight'] = np.exp(-np.log(2) * (max_idx - train['time_idx']) / HALF_LIFE_DAYS)
train['sample_weight'] = train['sample_weight'] / train['sample_weight'].mean()

print(f'Train: {len(train):,} rows, {train["ticker"].nunique()} tickers, → {train["Date"].max().date()}')
print(f'Val:   {len(val):,} rows, {val["ticker"].nunique()} tickers, {val["Date"].min().date()} → {val["Date"].max().date()}')
print(f'Test:  {len(test):,} rows, {test["ticker"].nunique()} tickers, {test["Date"].min().date()} → {test["Date"].max().date()}')

train.to_parquet(os.path.join(DRIVE_DATA, 'train.parquet'), index=False)
val.to_parquet(os.path.join(DRIVE_DATA, 'val.parquet'), index=False)
test.to_parquet(os.path.join(DRIVE_DATA, 'test.parquet'), index=False)

exclude = {'Date', 'ticker', 'sector', 'day_of_week', 'month', 'time_idx', 'sample_weight'}
target = 'Close'
time_varying_unknown_reals = [
    c for c in full_df.columns
    if c not in exclude and c != target
    and full_df[c].dtype in ['float64', 'float32', 'int64', 'int32']
]

actual_sectors = full_df.groupby('ticker')['sector'].first().to_dict()

updated_config = {
    'tickers': top_400,
    'sectors': actual_sectors,
    'target': target,
    'group_ids': ['ticker'],
    'static_categoricals': ['ticker', 'sector'],
    'time_varying_known_categoricals': ['day_of_week', 'month'],
    'time_varying_unknown_reals': time_varying_unknown_reals,
    'max_encoder_length': 60,
    'max_prediction_length': 22,
    'n_sentiment_components': config.get('n_sentiment_components', 32),
    'macro_cols': config.get('macro_cols', []),
    'table_metrics': config.get('table_metrics', []),
    'train_cutoff': '2025-06-30',
    'val_cutoff': '2025-10-31',
    'sample_weight_half_life': HALF_LIFE_DAYS,
}

with open(os.path.join(DRIVE_DATA, 'config.json'), 'w') as f:
    json.dump(updated_config, f, indent=2)

import shutil
shutil.copy2(os.path.join(DRIVE_DATA, 'config.json'), os.path.join(MODEL_DIR, 'config.json'))

print(f'\n=== Dataset Updated ===')
print(f'Features: {len(time_varying_unknown_reals)} time-varying unknown reals')
print(f'Tickers: {len(top_400)} (current S&P 500)')
print(f'Saved to: {DRIVE_DATA}')
print(f'Proceed to 03_model_training.ipynb for retraining')

Train: 2,316,737 rows, 400 tickers, → 2025-06-30
Val:   34,800 rows, 400 tickers, 2025-07-01 → 2025-10-31
Test:  41,600 rows, 400 tickers, 2025-11-03 → 2026-04-02

=== Dataset Updated ===
Features: 107 time-varying unknown reals
Tickers: 400 (current S&P 500)
Saved to: /content/drive/MyDrive/predictamarket/data
Proceed to 03_model_training.ipynb for retraining


## Summary

In [ ]:
print('=== Dataset Summary ===')
print(f'Total rows: {len(full_df):,}')
print(f'Tickers: {full_df["ticker"].nunique()}')
print(f'Date range: {full_df["Date"].min().date()} to {full_df["Date"].max().date()}')
print(f'Trading days: {full_df["time_idx"].nunique()}')
print()
print('=== Split ===')
print(f'Train: {len(train):,} ({len(train)/len(full_df)*100:.1f}%)')
print(f'Val:   {len(val):,} ({len(val)/len(full_df)*100:.1f}%)')
print(f'Test:  {len(test):,} ({len(test)/len(full_df)*100:.1f}%)')
print()
print('=== Close Price Stats ===')
print(f'Mean: ${full_df["Close"].mean():.2f}')
print(f'Median: ${full_df["Close"].median():.2f}')
print(f'NaN count: {full_df["Close"].isna().sum()}')
print(f'Inf count: {np.isinf(full_df["Close"]).sum()}')
print()
print('=== New Data Added ===')
new_rows = len(full_df[full_df['Date'] > pd.Timestamp('2024-10-31')])
print(f'Rows after Oct 2024: {new_rows:,}')
print(f'New months of data: {(full_df["Date"].max() - pd.Timestamp("2024-10-31")).days / 30:.0f}')

=== Dataset Summary ===
Total rows: 2,393,137
Tickers: 400
Date range: 2000-03-14 to 2026-04-02
Trading days: 6553

=== Split ===
Train: 2,316,737 (96.8%)
Val:   34,800 (1.5%)
Test:  41,600 (1.7%)

=== Close Price Stats ===
Mean: $73.78
Median: $31.45
NaN count: 0
Inf count: 0

=== New Data Added ===
Rows after Oct 2024: 141,600
New months of data: 17


In [ ]:
import pandas as pd
import numpy as np

train = pd.read_parquet('/content/drive/MyDrive/predictamarket/data/train.parquet')
val = pd.read_parquet('/content/drive/MyDrive/predictamarket/data/val.parquet')
test = pd.read_parquet('/content/drive/MyDrive/predictamarket/data/test.parquet')
full = pd.concat([train, val, test], ignore_index=True)
full['Date'] = pd.to_datetime(full['Date'])

print('=== 1. БАЗОВЫЕ ПРОВЕРКИ ===')
print(f'Строк: {len(full):,}')
print(f'NaN total: {full.isna().sum().sum()}')
print(f'Inf total: {np.isinf(full.select_dtypes(include=[np.number])).sum().sum()}')
print(f'Close <= 0: {(full["Close"] <= 0).sum()}')
print(f'Close NaN: {full["Close"].isna().sum()}')

print('\n=== 2. ДУБЛИКАТЫ ===')
dupes = full.duplicated(subset=['ticker', 'Date'], keep=False).sum()
print(f'Дубликатов (ticker + Date): {dupes}')

print('\n=== 3. ПЕРЕСЕЧЕНИЕ SPLITS ===')
train_dates = set(zip(train['ticker'], train['Date']))
val_dates = set(zip(val['ticker'], val['Date']))
test_dates = set(zip(test['ticker'], test['Date']))
print(f'Train ∩ Val: {len(train_dates & val_dates)}')
print(f'Train ∩ Test: {len(train_dates & test_dates)}')
print(f'Val ∩ Test: {len(val_dates & test_dates)}')

print('\n=== 4. ПОКРЫТИЕ ТИКЕРОВ ===')
print(f'Train tickers: {train["ticker"].nunique()}')
print(f'Val tickers: {val["ticker"].nunique()}')
print(f'Test tickers: {test["ticker"].nunique()}')
only_in_val = set(val['ticker'].unique()) - set(train['ticker'].unique())
only_in_test = set(test['ticker'].unique()) - set(train['ticker'].unique())
print(f'В val но не в train: {len(only_in_val)} {list(only_in_val)[:5]}')
print(f'В test но не в train: {len(only_in_test)} {list(only_in_test)[:5]}')

print('\n=== 5. НЕПРЕРЫВНОСТЬ ДАТ ===')
gaps = []
for ticker in full['ticker'].unique()[:50]:  # проверим 50 тикеров
    t = full[full['ticker'] == ticker].sort_values('Date')
    date_diff = t['Date'].diff().dt.days
    max_gap = date_diff.max()
    if max_gap > 5:  # больше 5 дней (выходные + праздники = норма до 4)
        gaps.append((ticker, int(max_gap)))
if gaps:
    print(f'Тикеры с пропусками > 5 дней: {len(gaps)}')
    for t, g in gaps[:10]:
        print(f'  {t}: max gap {g} дней')
else:
    print('Пропусков > 5 дней нет (первые 50 тикеров)')

print('\n=== 6. КЛЮЧЕВЫЕ ФИЧИ (нули в новом периоде) ===')
new_data = full[full['Date'] >= '2025-04-01']
key_features = ['vix', 'sp500', 'gold', 'treasury_10y', 'fed_funds_rate',
                'vix_contango', 'eps_surprise_pct', 'days_to_fomc',
                'sent_0', 'sent_1', 'news_count', 'insider_net',
                'log_return', 'rsi_14', 'ma_20', 'momentum_5d']

print(f'Строк после апреля 2025: {len(new_data)}')
for col in key_features:
    if col in new_data.columns:
        zero_pct = (new_data[col] == 0).mean() * 100
        nan_pct = new_data[col].isna().mean() * 100
        print(f'  {col:25s}: zeros={zero_pct:.1f}%, NaN={nan_pct:.1f}%, mean={new_data[col].mean():.4f}')
    else:
        print(f'  {col:25s}: MISSING')

print('\n=== 7. SAMPLE WEIGHT ===')
if 'sample_weight' in train.columns:
    print(f'Min: {train["sample_weight"].min():.4f}')
    print(f'Max: {train["sample_weight"].max():.4f}')
    print(f'Mean: {train["sample_weight"].mean():.4f}')
    print(f'NaN: {train["sample_weight"].isna().sum()}')
else:
    print('sample_weight отсутствует в train!')

del full, train, val, test
import gc; gc.collect()


=== 1. БАЗОВЫЕ ПРОВЕРКИ ===
Строк: 2,393,137
NaN total: 0
Inf total: 0
Close <= 0: 0
Close NaN: 0

=== 2. ДУБЛИКАТЫ ===
Дубликатов (ticker + Date): 0

=== 3. ПЕРЕСЕЧЕНИЕ SPLITS ===
Train ∩ Val: 0
Train ∩ Test: 0
Val ∩ Test: 0

=== 4. ПОКРЫТИЕ ТИКЕРОВ ===
Train tickers: 400
Val tickers: 400
Test tickers: 400
В val но не в train: 0 []
В test но не в train: 0 []

=== 5. НЕПРЕРЫВНОСТЬ ДАТ ===
Тикеры с пропусками > 5 дней: 40
  aapl: max gap 7 дней
  abt: max gap 7 дней
  acgl: max gap 7 дней
  adbe: max gap 7 дней
  adi: max gap 7 дней
  adm: max gap 7 дней
  adp: max gap 7 дней
  adsk: max gap 7 дней
  aee: max gap 7 дней
  aep: max gap 7 дней

=== 6. КЛЮЧЕВЫЕ ФИЧИ (нули в новом периоде) ===
Строк после апреля 2025: 101200
  vix                      : zeros=0.0%, NaN=0.0%, mean=19.4494
  sp500                    : zeros=0.0%, NaN=0.0%, mean=6441.2937
  gold                     : zeros=0.0%, NaN=0.0%, mean=3938.9715
  treasury_10y             : zeros=0.0%, NaN=0.0%, mean=4.2258
  fed_funds

325